# V5 · Resume XAI dari training yang sudah selesai
**Built with PriorLabs-TabPFN**

Notebook ini tidak menjalankan pencarian model, ekstraksi audio, atau training classifier. Ia memverifikasi checksum training yang selesai, memuat model/fitur tersimpan, memperbaiki evaluasi XAI yang sensitif terhadap batch, lalu melanjutkan robustness/cascade dan ekspor RETURN.

Kode training V5, bobot, cutoff, dan prediksi test disimpan utuh. Fallback XAI memakai inferensi satu baris dengan cache ketika prediksi batch berbeda, dengan pemeriksaan pointwise 1e-10 dan completeness 1e-8. Selisih terhadap prediksi test tersimpan dicatat pada sampel XAI, bukan dipakai untuk mengganti hasil test. Penyebab spesifik pada GPU belum dibuktikan tanpa keluaran run Anda.

Gunakan kernel V5 yang sama. Ekstrak seluruh ZIP perbaikan ke folder proyek, misalnya `D:/haqy/`, lalu buka notebook ini. Paket hanya menambahkan `src/v5_xai_resume.py`; jangan mengganti `coda_v5.py`, `coda_v5_models.py`, konfigurasi, atau RUN_NAME training.

In [ ]:
import os,sys,json
from pathlib import Path
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG",":4096:8")
os.environ.setdefault("HF_HUB_OFFLINE","1")
RUN_NAME = "experiment_v5_001"  # sama persis dengan training yang telah selesai
PACKAGE_DIR = ""  # jika perlu: r"<project>/research"
candidates=[Path(PACKAGE_DIR)] if PACKAGE_DIR else []
for parent in [Path.cwd(),*Path.cwd().parents]:candidates.extend([parent,parent/"research"])
PACKAGE=next((p.resolve() for p in candidates if (p/"src/v5_xai_resume.py").is_file() and (p/"config_v5.json").is_file()),None)
if PACKAGE is None:raise RuntimeError("Ekstrak ZIP perbaikan ke proyek V5, atau isi PACKAGE_DIR.")
sys.path.insert(0,str(PACKAGE))
from src import coda_v5 as v5
from src import coda_v5_plots as plots
from src import v5_xai_resume as repair
from IPython.display import display
import pandas as pd
cfg,table,splits,protected=repair.load_run(PACKAGE,RUN_NAME)
OUT=Path(cfg["output_dir"])
print("Run:",OUT,"| Peserta:",len(table),"| Fold:",len(splits["folds"]))
print("Training tetap digunakan; tidak ada pemanggilan inner_search/run_nested_cv.")

## Lanjutkan postprocessing
Fold dengan `POSTPROCESS_COMPLETE.json` yang valid digunakan kembali. Fold yang gagal menjalankan ulang postprocessing saja. Inferensi per baris dapat lebih lambat; cache mengurangi prediksi duplikat. Ini tidak mengulang training. Prediksi TabPFN dapat memulihkan konteks training tersimpan untuk inferensi, bukan melakukan hyperparameter search atau melatih bobot foundation model.

Kesalahan input atau prediksi pointwise yang tidak repeatable tetap menghentikan proses. Jangan menaikkan toleransi atau menghapus marker/checkpoint untuk melewatinya.

In [ ]:
try:
    v5.finish_postprocessing(table,splits,cfg)
finally:
    repair.verify_unchanged(cfg,protected)
diagnostics=repair.summarize(cfg)
display(pd.DataFrame([diagnostics]))
display(pd.read_csv(OUT/"cascade_pooled_counts.csv"))
display(pd.read_csv(OUT/"nested_individual_xai.csv").head(20))
plots.show(plots.postprocessing(cfg))

## Tinjau perbedaan inferensi dan hasil tersimpan
`rowwise_fallback_used` menandai fallback. `original_batch_raw_delta` merekam probe batch sebelum fallback; setelah fallback aktif, probe tidak diulang dan sebagian nilai dapat kosong. `archived_test_*_delta` serta `archived_youden_decision_changed` membandingkan endpoint penjelasan dengan prediksi test lama pada peserta sampel. Nilai besar/perubahan keputusan harus diaudit, bukan dihapus. Penjelasan memakai fungsi pointwise; hasil test lama tetap dipertahankan. Fold yang selesai sebelum patch dapat tidak memiliki diagnostik tambahan.

XAI belum divalidasi klinis; completeness adalah identitas matematis. Simulasi denyut bukan validasi kamera.

In [ ]:
display(pd.read_csv(OUT/"nested_grouped_xai.csv").head(20))
summary_path=OUT/"nested_mean_std.csv"
if summary_path.exists():
    display(pd.read_csv(summary_path)[["combination","policy","roc_auc_mean","roc_auc_std","sensitivity_mean","specificity_mean"]])
else:
    # Recompute reports only from already-frozen fold predictions, without fitting.
    metrics,predictions,summary,pooled,paired=v5.aggregate_results(splits,cfg)
    display(summary)

## Ekspor RETURN
Model, cutoff, dan seluruh artefak training yang dilindungi diperiksa lagi. RETURN menyertakan sumber patch, manifest diagnostik, dan salinan notebook resume untuk audit. Bawa ZIP hasil ini kembali ke laptop.

In [ ]:
repair.verify_unchanged(cfg,protected)
runtime=OUT/"runtime_sources"
for name in ["CODA_TB_V5_Resume_XAI.ipynb","V5_XAI_FIX_README.md"]:
    source=PACKAGE/name
    if source.exists():(runtime/name).write_bytes(source.read_bytes())
returned=v5.export_return(cfg,splits)
print("Selesai. Salin RETURN:",returned)